# Limpieza y preparación de datos complementarios — TFG (versión Colab)
**Beatriz Muñoz García-Serrano · Grado en Business Analytics · UFV**

**Versión adaptada a Google Colab.** Todos los archivos deben estar en `/content/`.

## Archivos que debes tener en `/content/` antes de ejecutar
| Archivo | Descripción |
|---|---|
| `gtrends_general.csv.csv` | Google Trends términos genéricos |
| `gtrends_estaciones1.csv.csv` | Google Trends estaciones principales |
| `gtrends_estaciones2.csv.csv` | Google Trends estaciones secundarias |

Los festivos y las orientaciones están hardcodeados (no necesitan archivo de entrada).

## Outputs generados en `/content/`
- `gtrends_clean.csv`
- `festivos_espana_2018_2026.csv`
- `orientacion_estaciones.csv`


In [1]:
import pandas as pd
import io
from datetime import date, timedelta

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


---
## 1. Google Trends

**Fuente:** Google Trends (trends.google.com)  
**Descarga:** manual, exportación CSV desde la plataforma  
**Configuración:** España, búsqueda web, enero 2018 – diciembre 2025  
**Escala:** índice relativo 0-100 (Google normaliza los valores; 100 = máximo interés en el período)

Se descargaron tres archivos:
- `gtrends_general.csv` — términos genéricos: *esquiar*, *estaciones de esquí*, *forfait esquí*
- `gtrends_estaciones1.csv` — estaciones principales: Baqueira Beret, Sierra Nevada, Formigal, Valdesquí, La Molina
- `gtrends_estaciones2.csv` — estaciones secundarias: Cerler, Boí Taüll, Masella, Astún, La Pinilla

**Limpieza aplicada:**
- Eliminación de comillas extra que Google añade al exportar
- Conversión de la columna de fecha a tipo datetime
- Renombrado de columnas a nombres limpios sin espacios ni tildes
- Filtrado al rango 2018-2025
- Unión de los tres archivos por fecha en un único dataframe

In [2]:
# Ruta a los archivos descargados (ajustar si es necesario)
BASE = '/content/'  # cambiar por la ruta donde estén los archivos

def leer_gtrends(path):
    """
    Lee un CSV de Google Trends y elimina las comillas extra
    que la plataforma añade al exportar.
    """
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    content = content.replace('"', '')  # quitar comillas extra
    df = pd.read_csv(io.StringIO(content))
    df.rename(columns={'Time': 'fecha'}, inplace=True)
    df['fecha'] = pd.to_datetime(df['fecha'])
    return df

# Cargar los tres archivos
df_gen  = leer_gtrends(BASE + 'gtrends_general.csv.csv')
df_est1 = leer_gtrends(BASE + 'gtrends_estaciones1.csv.csv')
df_est2 = leer_gtrends(BASE + 'gtrends_estaciones2.csv.csv')

# Renombrar columnas a nombres limpios
df_gen.columns  = ['fecha', 'gt_esquiar', 'gt_estaciones_esqui', 'gt_forfait_esqui']
df_est1.columns = ['fecha', 'gt_baqueira', 'gt_sierra_nevada', 'gt_formigal',
                   'gt_valdesque', 'gt_la_molina']
df_est2.columns = ['fecha', 'gt_cerler', 'gt_boi_taull', 'gt_masella',
                   'gt_astun', 'gt_la_pinilla']

# Unir los tres por fecha
df_gt = df_gen.merge(df_est1, on='fecha', how='outer') \
              .merge(df_est2, on='fecha', how='outer')

# Filtrar al rango de análisis
df_gt = df_gt[(df_gt['fecha'] >= '2018-01-01') & (df_gt['fecha'] <= '2025-12-31')]
df_gt = df_gt.sort_values('fecha').reset_index(drop=True)

# Añadir columnas de año y mes para facilitar joins posteriores
df_gt['anio'] = df_gt['fecha'].dt.year
df_gt['mes']  = df_gt['fecha'].dt.month

print(f'Registros: {len(df_gt)}')
print(f'Período: {df_gt["fecha"].min().date()} → {df_gt["fecha"].max().date()}')
print(f'Nulos:\n{df_gt.isnull().sum()}')
df_gt.head()

Registros: 96
Período: 2018-01-01 → 2025-12-01
Nulos:
fecha                  0
gt_esquiar             0
gt_estaciones_esqui    0
gt_forfait_esqui       0
gt_baqueira            0
gt_sierra_nevada       0
gt_formigal            0
gt_valdesque           0
gt_la_molina           0
gt_cerler              0
gt_boi_taull           0
gt_masella             0
gt_astun               0
gt_la_pinilla          0
anio                   0
mes                    0
dtype: int64


,fecha,gt_esquiar,gt_estaciones_esqui,gt_forfait_esqui,gt_baqueira,gt_sierra_nevada,gt_formigal,gt_valdesque,gt_la_molina,gt_cerler,gt_boi_taull,gt_masella,gt_astun,gt_la_pinilla,anio,mes
0,2018-01-01,73,2,1,44,51,5,23,25,35,0,32,32,39,2018,1
1,2018-02-01,66,2,1,46,47,9,23,31,37,0,39,32,31,2018,2
2,2018-03-01,34,1,0,30,81,14,17,20,29,0,24,25,21,2018,3
3,2018-04-01,12,1,0,7,50,4,12,5,7,0,13,10,8,2018,4
4,2018-05-01,5,0,0,2,16,1,1,3,2,0,3,2,3,2018,5


In [3]:
# Guardar
df_gt.to_csv('/content/gtrends_clean.csv', index=False, encoding='utf-8-sig')
print('Guardado: datos/clean/gtrends_clean.csv')

Guardado: datos/clean/gtrends_clean.csv


---
## 2. Calendario de festivos (2018-2026)

**Fuente:** BOE (Boletín Oficial del Estado) — festivos nacionales y autonómicos  
**Construcción:** elaboración propia a partir de las fechas publicadas cada año  

Se incluyen:
- Festivos **nacionales** (comunes a toda España)
- Festivos **autonómicos** de las comunidades con estaciones de esquí en el dataset:
  Madrid, Andalucía, Aragón y Cataluña

**Variables creadas:**
- `temporada_esqui`: 1 si el festivo cae en noviembre-abril
- `semana_navidad`: 1 si cae entre el 24 de diciembre y el 7 de enero
- `semana_santa`: 1 si cae en los 3 días antes o después del Viernes Santo

In [4]:
# Fechas de Viernes Santo (variable cada año)
# Fuente: BOE y calendarios litúrgicos oficiales
semana_santa_viernes = {
    2018: '2018-03-30', 2019: '2019-04-19', 2020: '2020-04-10',
    2021: '2021-04-02', 2022: '2022-04-15', 2023: '2023-04-07',
    2024: '2024-03-29', 2025: '2025-04-18', 2026: '2026-04-03',
}
semana_santa_jueves = {
    2018: '2018-03-29', 2019: '2019-04-18', 2020: '2020-04-09',
    2021: '2021-04-01', 2022: '2022-04-14', 2023: '2023-04-06',
    2024: '2024-03-28', 2025: '2025-04-17', 2026: '2026-04-02',
}

# Festivos fijos nacionales
fijos_nacionales = [
    ('01-01', 'Año Nuevo'),
    ('01-06', 'Epifanía del Señor (Reyes)'),
    ('05-01', 'Día del Trabajo'),
    ('08-15', 'Asunción de la Virgen'),
    ('10-12', 'Fiesta Nacional de España'),
    ('11-01', 'Todos los Santos'),
    ('12-06', 'Día de la Constitución'),
    ('12-08', 'Inmaculada Concepción'),
    ('12-25', 'Navidad'),
]

# Festivos autonómicos relevantes
auto_list = [
    ('05-02', 'Comunidad de Madrid',       'autonomico', 'Madrid'),
    ('02-28', 'Día de Andalucía',           'autonomico', 'Andalucia'),
    ('04-23', 'Día de Aragón (San Jorge)',  'autonomico', 'Aragon'),
    ('09-11', 'Diada Nacional de Catalunya','autonomico', 'Cataluna'),
    ('12-26', 'San Esteban (Sant Esteve)',  'autonomico', 'Cataluna'),
]

festivos = []

for year in range(2018, 2027):
    # Nacionales fijos
    for mm_dd, nombre in fijos_nacionales:
        festivos.append({'fecha': f'{year}-{mm_dd}', 'nombre': nombre,
                         'tipo': 'nacional', 'comunidades': 'todas'})
    # Semana Santa (nacional: solo Viernes Santo)
    festivos.append({'fecha': semana_santa_viernes[year], 'nombre': 'Viernes Santo',
                     'tipo': 'nacional', 'comunidades': 'todas'})
    # Jueves Santo (Madrid, Andalucía, Aragón, Cataluña)
    festivos.append({'fecha': semana_santa_jueves[year], 'nombre': 'Jueves Santo',
                     'tipo': 'autonomico', 'comunidades': 'Madrid, Andalucia, Aragon, Cataluna'})
    # Autonómicos
    for mm_dd, nombre, tipo, ccaa in auto_list:
        festivos.append({'fecha': f'{year}-{mm_dd}', 'nombre': nombre,
                         'tipo': tipo, 'comunidades': ccaa})

df_fest = pd.DataFrame(festivos)
df_fest['fecha'] = pd.to_datetime(df_fest['fecha'])
df_fest = df_fest.sort_values('fecha').reset_index(drop=True)

# Flags de período
df_fest['temporada_esqui'] = df_fest['fecha'].dt.month.isin([11,12,1,2,3,4]).astype(int)

def es_navidad(d):
    return (d.month == 12 and d.day >= 24) or (d.month == 1 and d.day <= 7)
df_fest['semana_navidad'] = df_fest['fecha'].apply(es_navidad).astype(int)

ss_ext = set()
for f in pd.to_datetime(list(semana_santa_viernes.values())):
    for delta in range(-3, 4):
        ss_ext.add(f + timedelta(days=delta))
df_fest['semana_santa'] = df_fest['fecha'].isin(ss_ext).astype(int)

print(f'Total festivos: {len(df_fest)}')
print(f'En temporada de esquí: {df_fest["temporada_esqui"].sum()}')
df_fest.head(10)

Total festivos: 144
En temporada de esquí: 99


,fecha,nombre,tipo,comunidades,temporada_esqui,semana_navidad,semana_santa
0,2018-01-01,Año Nuevo,nacional,todas,1,1,0
1,2018-01-06,Epifanía del Señor (Reyes),nacional,todas,1,1,0
2,2018-02-28,Día de Andalucía,autonomico,Andalucia,1,0,0
3,2018-03-29,Jueves Santo,autonomico,"Madrid, Andalucia, Aragon, Cataluna",1,0,1
4,2018-03-30,Viernes Santo,nacional,todas,1,0,1
5,2018-04-23,Día de Aragón (San Jorge),autonomico,Aragon,1,0,0
6,2018-05-01,Día del Trabajo,nacional,todas,0,0,0
7,2018-05-02,Comunidad de Madrid,autonomico,Madrid,0,0,0
8,2018-08-15,Asunción de la Virgen,nacional,todas,0,0,0
9,2018-09-11,Diada Nacional de Catalunya,autonomico,Cataluna,0,0,0


In [5]:
df_fest.to_csv('/content/festivos_espana_2018_2026.csv', index=False, encoding='utf-8-sig')
print('Guardado: datos/clean/festivos_espana_2018_2026.csv')

Guardado: datos/clean/festivos_espana_2018_2026.csv


---
## 3. Características de estaciones de esquí

**Fuente:** fichas técnicas oficiales de cada estación y verificación con fuentes secundarias  
- Valdesquí: [valdesqui.es](https://valdesqui.es/en/map-and-technical-sheet/) y Wikipedia  
- Sierra Nevada: [sierranevada.es](https://sierranevada.es) y granadadirect.com  
- Baqueira, Boí Taüll, Masella, La Molina: [esqui.com](https://www.esqui.com)  
- Resto de estaciones pirenaicas y cantábricas: cartografía IGN y documentación sectorial ATUDEM

**Variables incluidas:**
- `orientacion`: orientación predominante de las pistas (N, NE, S...)
- `altitud_base_m` y `altitud_cima_m`: cotas de la estación en metros
- `comunidad`: comunidad autónoma
- `orientacion_sol`: descripción del efecto de la orientación sobre la nieve

**Nota metodológica:** la orientación recoge la ladera predominante de cada estación.
Sierra Nevada es la única estación con orientación sur (solana), lo que condiciona
la calidad y durabilidad de su nieve respecto al resto.

In [6]:
# Datos verificados con fuentes oficiales de cada estación
# Orientación: N = umbría (norte), S = solana (sur), NE = noreste
orientaciones = [
    {'estacion': 'Alto Campoo',          'orientacion': 'N',  'altitud_base_m': 1530, 'altitud_cima_m': 2175, 'comunidad': 'Cantabria'},
    {'estacion': 'Astún',                'orientacion': 'N',  'altitud_base_m': 1700, 'altitud_cima_m': 2300, 'comunidad': 'Aragón'},
    {'estacion': 'Baqueira Beret',       'orientacion': 'N',  'altitud_base_m': 1500, 'altitud_cima_m': 2610, 'comunidad': 'Cataluña'},
    {'estacion': 'Boí Taüll',            'orientacion': 'N',  'altitud_base_m': 2020, 'altitud_cima_m': 2751, 'comunidad': 'Cataluña'},
    {'estacion': 'Candanchú',            'orientacion': 'N',  'altitud_base_m': 1530, 'altitud_cima_m': 2400, 'comunidad': 'Aragón'},
    {'estacion': 'Cerler',               'orientacion': 'NE', 'altitud_base_m': 1550, 'altitud_cima_m': 2630, 'comunidad': 'Aragón'},
    {'estacion': 'Espot',                'orientacion': 'N',  'altitud_base_m': 1540, 'altitud_cima_m': 2520, 'comunidad': 'Cataluña'},
    {'estacion': 'Formigal',             'orientacion': 'N',  'altitud_base_m': 1510, 'altitud_cima_m': 2250, 'comunidad': 'Aragón'},
    {'estacion': 'Fuentes de Invierno',  'orientacion': 'N',  'altitud_base_m': 1650, 'altitud_cima_m': 1920, 'comunidad': 'Asturias'},
    {'estacion': 'Javalambre',           'orientacion': 'N',  'altitud_base_m': 1730, 'altitud_cima_m': 2020, 'comunidad': 'Aragón'},
    {'estacion': 'La Molina',            'orientacion': 'N',  'altitud_base_m': 1700, 'altitud_cima_m': 2537, 'comunidad': 'Cataluña'},
    {'estacion': 'La Pinilla',           'orientacion': 'N',  'altitud_base_m': 1500, 'altitud_cima_m': 2250, 'comunidad': 'Castilla y León'},
    {'estacion': 'Manzaneda',            'orientacion': 'N',  'altitud_base_m': 1430, 'altitud_cima_m': 1780, 'comunidad': 'Galicia'},
    {'estacion': 'Masella',              'orientacion': 'N',  'altitud_base_m': 1600, 'altitud_cima_m': 2535, 'comunidad': 'Cataluña'},
    {'estacion': 'Panticosa',            'orientacion': 'N',  'altitud_base_m': 1520, 'altitud_cima_m': 2200, 'comunidad': 'Aragón'},
    {'estacion': 'Port Ainé',            'orientacion': 'N',  'altitud_base_m': 1650, 'altitud_cima_m': 2440, 'comunidad': 'Cataluña'},
    {'estacion': 'Port del Comte',       'orientacion': 'N',  'altitud_base_m': 1700, 'altitud_cima_m': 2390, 'comunidad': 'Cataluña'},
    {'estacion': 'Puerto de Navacerrada','orientacion': 'N',  'altitud_base_m': 1800, 'altitud_cima_m': 2200, 'comunidad': 'Madrid'},
    {'estacion': 'San Isidro',           'orientacion': 'N',  'altitud_base_m': 1520, 'altitud_cima_m': 2100, 'comunidad': 'Castilla y León'},
    {'estacion': 'Sierra Nevada',        'orientacion': 'S',  'altitud_base_m': 2100, 'altitud_cima_m': 3300, 'comunidad': 'Andalucía'},
    {'estacion': 'Sierra de Béjar',      'orientacion': 'N',  'altitud_base_m': 1450, 'altitud_cima_m': 2050, 'comunidad': 'Castilla y León'},
    {'estacion': 'Tavascán',             'orientacion': 'N',  'altitud_base_m': 1640, 'altitud_cima_m': 2250, 'comunidad': 'Cataluña'},
    {'estacion': 'Valdelinares',         'orientacion': 'N',  'altitud_base_m': 1700, 'altitud_cima_m': 2020, 'comunidad': 'Aragón'},
    {'estacion': 'Valdesquí',            'orientacion': 'N',  'altitud_base_m': 1860, 'altitud_cima_m': 2280, 'comunidad': 'Madrid'},
    {'estacion': 'Valdezcaray',          'orientacion': 'N',  'altitud_base_m': 1350, 'altitud_cima_m': 2270, 'comunidad': 'La Rioja'},
    {'estacion': 'Valgrande-Pajares',    'orientacion': 'N',  'altitud_base_m': 1600, 'altitud_cima_m': 1850, 'comunidad': 'Asturias'},
    {'estacion': 'Vall de Nuria',        'orientacion': 'N',  'altitud_base_m': 1964, 'altitud_cima_m': 2252, 'comunidad': 'Cataluña'},
    {'estacion': 'Valle Laciana - Leitariegos', 'orientacion': 'N', 'altitud_base_m': 1520, 'altitud_cima_m': 1925, 'comunidad': 'Castilla y León'},
    {'estacion': 'Vallter',              'orientacion': 'N',  'altitud_base_m': 1820, 'altitud_cima_m': 2535, 'comunidad': 'Cataluña'},
    {'estacion': 'Punto de Nieve Santa Inés', 'orientacion': 'N', 'altitud_base_m': 1650, 'altitud_cima_m': 1900, 'comunidad': 'Castilla y León'},
]

df_ori = pd.DataFrame(orientaciones)

# Variable derivada: efecto de la orientación sobre la calidad de nieve
df_ori['umbria'] = (df_ori['orientacion'] == 'N').astype(int)  # 1=umbría, 0=solana/mixta

# Desnivel
df_ori['desnivel_m'] = df_ori['altitud_cima_m'] - df_ori['altitud_base_m']

print(f'Estaciones: {len(df_ori)}')
print(f'Orientación N (umbría): {(df_ori["orientacion"]=="N").sum()}')
print(f'Orientación S (solana): {(df_ori["orientacion"]=="S").sum()}')
print(f'Orientación NE: {(df_ori["orientacion"]=="NE").sum()}')
df_ori

Estaciones: 30
Orientación N (umbría): 28
Orientación S (solana): 1
Orientación NE: 1


,estacion,orientacion,altitud_base_m,altitud_cima_m,comunidad,umbria,desnivel_m
0,Alto Campoo,N,1530,2175,Cantabria,1,645
1,Astún,N,1700,2300,Aragón,1,600
2,Baqueira Beret,N,1500,2610,Cataluña,1,1110
3,Boí Taüll,N,2020,2751,Cataluña,1,731
4,Candanchú,N,1530,2400,Aragón,1,870
5,Cerler,NE,1550,2630,Aragón,0,1080
6,Espot,N,1540,2520,Cataluña,1,980
7,Formigal,N,1510,2250,Aragón,1,740
8,Fuentes de Invierno,N,1650,1920,Asturias,1,270
9,Javalambre,N,1730,2020,Aragón,1,290


In [7]:
df_ori.to_csv('/content/orientacion_estaciones.csv', index=False, encoding='utf-8-sig')
print('Guardado: datos/clean/orientacion_estaciones.csv')

Guardado: datos/clean/orientacion_estaciones.csv


---
## Resumen de outputs

| Archivo | Registros | Variables | Descripción |
|---|---|---|---|
| `gtrends_clean.csv` | 96 (mensual) | 14 | Interés de búsqueda Google Trends |
| `festivos_espana_2018_2026.csv` | 144 | 7 | Festivos nacionales y autonómicos |
| `orientacion_estaciones.csv` | 30 | 6 | Características físicas de estaciones |

Todos los archivos están guardados en `datos/clean/` y listos para la fase de integración.

In [8]:
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/datos_complementarios_limpios.zip', 'w') as z:
    z.write('/content/gtrends_clean.csv', arcname='gtrends_clean.csv')
    z.write('/content/festivos_espana_2018_2026.csv', arcname='festivos_espana_2018_2026.csv')
    z.write('/content/orientacion_estaciones.csv', arcname='orientacion_estaciones.csv')

files.download('/content/datos_complementarios_limpios.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>